In [1]:
from pathlib import Path
from config import DATA_PATH_RAW, LABELS_SUBDIR, IMAGES_SUBDIR, ALFS_SUBDIR, THERMAL_SUBDIR

SPLITS = ["train", "val", "test"]


# class mapping
CLASS_NAMES = {
    0: "animal",
    # 1: "red_deer",
    # 2: "roe_deer",
    # 3: "chamois",
    # 4: "human",
    # 5: "alpine_ibex",
    # 6: "fallow_deer",
    # 7: "unknown",
    # 8: "dog",
    # 9: "bird",
    # 10: "wild_boar",
    # 11: "hybrid_pig"
}

In [2]:
# Imagesize
from PIL import Image
import os

img_path = DATA_PATH_RAW / THERMAL_SUBDIR / IMAGES_SUBDIR / 'train'

for f in os.listdir(img_path):
    if f.endswith((".jpg")):
        img = Image.open(os.path.join(img_path, f))
        print(f"name: {f}, size (w/h): {img.size}")
        break

name: 0_8082.jpg, size (w/h): (1024, 1024)


In [3]:
from pathlib import Path
from collections import Counter
import numpy as np
import csv

images_total = 0
images_with_animals = 0
images_without_animals = 0

animals_per_image = []

class_counter = Counter()

In [4]:
for split in SPLITS:
    label_dir = DATA_PATH_RAW / THERMAL_SUBDIR / LABELS_SUBDIR / split
    if not label_dir.exists():
        continue

    for file in label_dir.glob("*.txt"):
        images_total += 1

        with open(file, "r") as f:
            lines = [l.strip() for l in f if l.strip()]

        # Case: empty or only "0" → no animal image
        if len(lines) == 0 or (len(lines) == 1 and lines[0].split()[0] == "0"):
            images_without_animals += 1
            animals_per_image.append(0)
            continue

        # image contains animals
        images_with_animals += 1
        animals_per_image.append(len(lines))

        for line in lines:
            cls = int(line.split()[0])


            class_counter[cls] += 1

total_animals = sum(class_counter.values())
avg_animals_per_image = total_animals / images_total if images_total else 0

animals_array = np.array(animals_per_image)

In [5]:
print(f"Total images: {images_total}")
print(f"Images with animals: {images_with_animals}")
print(f"Images without animals: {images_without_animals}")
print(f"Percentage with animals: {images_with_animals / images_total * 100:.2f}%")

print("\n--- Animal stats ---")
print(f"Total animals: {total_animals}")
print(f"Average animals per image: {avg_animals_per_image:.2f}")
print(f"Max animals in one image: {animals_array.max()}")

print(f"95th percentile animals/image: {np.percentile(animals_array, 95):.2f}")
print(f"99th percentile animals/image: {np.percentile(animals_array, 99):.2f}")

print("\n--- Per class distribution ---")

all_class_ids = sorted(CLASS_NAMES.keys())

for cls_id in all_class_ids:
    name = CLASS_NAMES[cls_id]
    count = class_counter.get(cls_id, 0)

    print(f"{cls_id:2d} {name:15s}: {count}")


Total images: 17947
Images with animals: 10917
Images without animals: 7030
Percentage with animals: 60.83%

--- Animal stats ---
Total animals: 47777
Average animals per image: 2.66
Max animals in one image: 36
95th percentile animals/image: 9.00
99th percentile animals/image: 15.00

--- Per class distribution ---
 0 animal         : 47777


### Class occurrences per split

Compare box counts per class across train / val / test. A large mismatch between splits (e.g. a class only in val) will hurt training and make mAP misleading.

In [6]:
IMAGE_SIZE = 1024

def collect_box_sizes(label_dir):
    widths_px, heights_px = [], []
    widths_norm, heights_norm = [], []

    for file in label_dir.glob("*.txt"):
        for line in file.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            w, h = map(float, parts[3:5])
            widths_norm.append(w)
            heights_norm.append(h)
            widths_px.append(w * IMAGE_SIZE)
            heights_px.append(h * IMAGE_SIZE)

    return widths_px, heights_px, widths_norm, heights_norm

def print_size_stats(name, w_px, h_px, w_norm, h_norm):
    if not w_px:
        print(f"{name}: no boxes")
        return

    w_px, h_px = np.array(w_px), np.array(h_px)
    w_norm, h_norm = np.array(w_norm), np.array(h_norm)

    print(f"\n{name} ({len(w_px)} boxes)")
    print(
        f"  width  | min={w_px.min():6.1f}px ({w_norm.min():.4f})  "
        f"avg={w_px.mean():6.1f}px ({w_norm.mean():.4f})  "
        f"max={w_px.max():6.1f}px ({w_norm.max():.4f})"
    )
    print(
        f"  height | min={h_px.min():6.1f}px ({h_norm.min():.4f})  "
        f"avg={h_px.mean():6.1f}px ({h_norm.mean():.4f})  "
        f"max={h_px.max():6.1f}px ({h_norm.max():.4f})"
    )
    min_side = np.minimum(w_px, h_px)
    print(
        f"  min side (min(w,h)) | min={min_side.min():6.1f}px  "
        f"avg={min_side.mean():6.1f}px  max={min_side.max():6.1f}px"
    )

all_w_px, all_h_px, all_w_n, all_h_n = [], [], [], []

print("--- Box size stats per split ---")
for split in SPLITS:
    label_dir = DATA_PATH_RAW / THERMAL_SUBDIR / LABELS_SUBDIR / split
    if not label_dir.exists():
        continue

    w_px, h_px, w_n, h_n = collect_box_sizes(label_dir)
    print_size_stats(split.upper(), w_px, h_px, w_n, h_n)

    all_w_px.extend(w_px)
    all_h_px.extend(h_px)
    all_w_n.extend(w_n)
    all_h_n.extend(h_n)

print("\n--- Box size stats (all splits) ---")
print_size_stats("ALL", all_w_px, all_h_px, all_w_n, all_h_n)

--- Box size stats per split ---

TRAIN (54347 boxes)
  width  | min=   1.0px (0.0010)  avg=  41.7px (0.0408)  max=1023.0px (0.9990)
  height | min=   1.0px (0.0010)  avg=  41.3px (0.0403)  max= 948.0px (0.9258)
  min side (min(w,h)) | min=   1.0px  avg=  35.7px  max=  93.7px

VAL (460 boxes)
  width  | min=  14.7px (0.0144)  avg=  34.9px (0.0341)  max=  94.2px (0.0920)
  height | min=   2.0px (0.0020)  avg=  44.2px (0.0432)  max=  74.2px (0.0725)
  min side (min(w,h)) | min=   2.0px  avg=  32.2px  max=  55.8px

--- Box size stats (all splits) ---

ALL (54807 boxes)
  width  | min=   1.0px (0.0010)  avg=  41.7px (0.0407)  max=1023.0px (0.9990)
  height | min=   1.0px (0.0010)  avg=  41.3px (0.0403)  max= 948.0px (0.9258)
  min side (min(w,h)) | min=   1.0px  avg=  35.7px  max=  93.7px


In [7]:
class_counter_per_split = {split: Counter() for split in SPLITS}

for split in SPLITS:
    label_dir = DATA_PATH_RAW / THERMAL_SUBDIR / LABELS_SUBDIR / split
    if not label_dir.exists():
        continue

    for file in label_dir.glob("*.txt"):
        with open(file, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                cls = int(line.split()[0])
                class_counter_per_split[split][cls] += 1

# table: box counts per class per split
print("--- Per-class occurrences per split (box counts) ---\n")
col_w = 10
header = f"{'id':>3} {'class':15s}" + "".join(f"{s:>{col_w}s}" for s in SPLITS)
print(header)
print("-" * len(header))

for cls_id in sorted(CLASS_NAMES.keys()):
    counts = [class_counter_per_split[s].get(cls_id, 0) for s in SPLITS]
    if sum(counts) == 0:
        continue
    print(
        f"{cls_id:3d} {CLASS_NAMES[cls_id]:15s}"
        + "".join(f"{c:>{col_w}d}" for c in counts)
    )

print("-" * len(header))
split_totals = [sum(class_counter_per_split[s].values()) for s in SPLITS]
print(f"{'':3} {'TOTAL':15s}" + "".join(f"{t:>{col_w}d}" for t in split_totals))

# percentage within each split
print("\n--- Per-class share within each split (%) ---\n")
print(header)
print("-" * len(header))

for cls_id in sorted(CLASS_NAMES.keys()):
    pcts = []
    has_any = False
    for split in SPLITS:
        total = split_totals[SPLITS.index(split)] or 1
        count = class_counter_per_split[split].get(cls_id, 0)
        pcts.append(count / total * 100)
        has_any |= count > 0
    if not has_any:
        continue
    print(
        f"{cls_id:3d} {CLASS_NAMES[cls_id]:15s}"
        + "".join(f"{p:>{col_w}.1f}" for p in pcts)
    )

--- Per-class occurrences per split (box counts) ---

 id class               train       val      test
-------------------------------------------------
  0 animal              54347       460         0
-------------------------------------------------
    TOTAL               54347       460         0

--- Per-class share within each split (%) ---

 id class               train       val      test
-------------------------------------------------
  0 animal              100.0     100.0       0.0


In [8]:
print("\n--- Class imbalance (percentage, FULL) ---")

all_class_ids = sorted(CLASS_NAMES.keys())

for cls_id in all_class_ids:
    count = class_counter.get(cls_id, 0)
    pct = (count / total_animals) * 100 if total_animals else 0

    print(f"{CLASS_NAMES[cls_id]:15s}: {pct:.2f}%")


--- Class imbalance (percentage, FULL) ---
animal         : 100.00%


In [9]:
print("\n--- Rare classes (<5%) ---")

for cls_id in all_class_ids:
    count = class_counter.get(cls_id, 0)
    pct = (count / total_animals) * 100 if total_animals else 0

    if pct < 5:
        print(f"{CLASS_NAMES[cls_id]:15s}: {count:5d} - {pct:.2f}%")


--- Rare classes (<5%) ---


In [10]:
from collections import Counter as C
dist = C(animals_per_image)

print("--- Animals per image distribution ---")
for k in sorted(dist):
    print(f"{k} animals: {dist[k]} images")

--- Animals per image distribution ---
0 animals: 7030 images
2 animals: 4025 images
3 animals: 2138 images
4 animals: 1251 images
5 animals: 931 images
6 animals: 624 images
7 animals: 460 images
8 animals: 355 images
9 animals: 274 images
10 animals: 220 images
11 animals: 163 images
12 animals: 141 images
13 animals: 79 images
14 animals: 69 images
15 animals: 35 images
16 animals: 33 images
17 animals: 20 images
18 animals: 18 images
19 animals: 9 images
20 animals: 12 images
21 animals: 7 images
22 animals: 4 images
23 animals: 4 images
24 animals: 8 images
25 animals: 6 images
26 animals: 10 images
27 animals: 4 images
28 animals: 1 images
29 animals: 3 images
30 animals: 1 images
31 animals: 3 images
32 animals: 3 images
33 animals: 2 images
34 animals: 1 images
35 animals: 1 images
36 animals: 2 images


In [11]:
with open("analysis/dataset_class_distribution.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["class_id", "class_name", "count", "percentage"])

    for cls_id, count in sorted(class_counter.items()):
        pct = (count / total_animals) * 100 if total_animals else 0
        writer.writerow([cls_id, CLASS_NAMES.get(cls_id), count, pct])

print("\nCSV saved: dataset_class_distribution.csv")


CSV saved: dataset_class_distribution.csv


In [ ]:
# look at black and blurred images -> count:
from utils.preprocess_methods import is_mostly_black, is_blurry

from pathlib import Path

def scan_quality_stats(base_path: Path, splits):
    stats = {}

    for split in splits:
        img_dir = base_path / IMAGES_SUBDIR / split

        total = 0
        black = 0
        blurry = 0

        for img_path in img_dir.glob("*.jpg"):
            total += 1

            if is_mostly_black(img_path):
                black += 1
                continue

            if is_blurry(img_path):
                blurry += 1
                continue

        stats[split] = {
            "total": total,
            "black": black,
            "blurry": blurry,
            "kept": total - black - blurry
        }

    return stats

In [14]:
raw_stats = scan_quality_stats(DATA_PATH_RAW / THERMAL_SUBDIR, SPLITS)

print("\n===== QUALITY CHECK (RAW DATA) =====")
for split, s in raw_stats.items():
    print(
        f"{split.upper():5s} | "
        f"total={s['total']:5d} | "
        f"kept={s['kept']:5d} | "
        f"black={s['black']:5d} | "
        f"blurry={s['blurry']:5d}"
    )


===== QUALITY CHECK (RAW DATA) =====
TRAIN | total=19173 | kept= 8965 | black=    0 | blurry=10208
VAL   | total= 2492 | kept= 1307 | black=    0 | blurry= 1185
TEST  | total= 2490 | kept= 2055 | black=    0 | blurry=  435
